# AXI test 21: the serving side on a bigger model, counted the way a company that runs the model counts
**Why this test exists.** The first seven runs used a 7B model on a Tesla T4. This run uses **Qwen2.5-14B-Instruct, 4-bit AWQ** (twice the size) on an **NVIDIA L4** and measures what a company that runs a model counts: **drift** (does the held state stay exact over 100 steps), **speed** (seconds per step, exact steps per GPU hour), **latency** (does a step get slower as the job goes on), **tokens** (computed prompt tokens per step) and **energy** (the GPU board's joules per exact step, read with `nvidia-smi` every half second). Seven cells on the sixth run's jobs, no retry anywhere: `P0`, `PN` (32 jobs: the capped grammar and the grammar that names the coordinates), `P4`, `PC` (4 jobs: the exact state and the whole chat kept), and `X0`, `XW`, `XR` (32 jobs: the request with coordinates plus a list of names, in ordinary words with the model reading the name, in ordinary words with the host reading the sentence). The predictions and the reading rules were written before this run in `docs/PROVIDER_SIDE_BIG_TEST.md`.

Engine **vLLM 0.31.0**, cache fixed at **32,000 tokens**, GPU **L4**.

**Before you start:** Runtime menu -> *Change runtime type* -> **L4 GPU** -> Save. **Choose the L4 and nothing else:** the pre-flight check says RED on any other card. Run each cell **one at a time, top to bottom**, wait for the green tick before the next one. **Keep this browser tab open** while a cell runs.

**Cost:** about 1.7 compute units an hour on an L4 (check *Runtime -> View resources* in your app). The whole run is about 2 to 3.5 hours (installing the engine about 5 minutes, loading the model 8 to 15 minutes, the pre-flight and safety check 3 minutes, the seven cells 60 to 120 minutes), so about 4 to 6 units. If you want to stop, you can stop after any cell: the finished cells are saved to Drive.

**If Colab stops** (the tab closes, the GPU is taken away): reconnect, run Cells 1 to 6 again (do not forget Cell 4b), and then run the cell you were in. Finished cells are skipped; a cell that was cut off starts again from its beginning.

Do not paste any keys anywhere.

### Cell 1: check the GPU (it must say NVIDIA L4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: get the code from GitHub. If it stops with 'the code is not on GitHub yet', finish the update 71 commands in Codespaces (git push) and run this cell again

In [ ]:
!rm -rf /content/axi && git clone -q --depth 1 https://github.com/gaurshalendra2412/axi-runtime /content/axi
import os
assert os.path.exists("/content/axi/python/axi/experiments/provider_side_big.py"), "the code is not on GitHub yet: finish the update 71 commands (git push) first"
assert os.path.exists("/content/axi/python/axi/experiments/provider_side_long.py"), "the code is not on GitHub yet: finish the update 71 commands (git push) first"
assert os.path.exists("/content/axi/python/axi/experiments/provider_side_typed.py"), "the code is not on GitHub yet: finish the update 71 commands (git push) first"
assert os.path.exists("/content/axi/python/axi/experiments/q_measure.py"), "the code is not on GitHub yet: finish the update 71 commands (git push) first"
print("code ok")

### Cell 3: install the engine, version 0.31.0 (about 5 minutes). If Colab says 'Restart session', press it, then run Cells 2, 3, 4 and 4b again (Cell 3 will then be quick)

In [ ]:
!pip -q install vllm==0.31.0

### Cell 4: connect Google Drive (a pop-up asks permission: tap Connect / Allow). If it prints 'Drive NOT connected', the run still works but nothing survives a disconnect

In [ ]:
import sys, os, json, time, glob, re
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_provider_big")

### Cell 4b: fix the output stream (about 1 second). vLLM 0.31.0 asks the notebook for a file number of its output and a Colab cell has none, so Cell 6 stops with `io.UnsupportedOperation: fileno`. This cell gives the notebook's output the usual numbers 1 and 2. Run it once after every restart, before Cell 6

In [ ]:
import sys
for name, fd in (("stdout", 1), ("stderr", 2)):
    s = getattr(sys, name)
    try: s.fileno()
    except Exception: s.fileno = (lambda fd=fd: fd)
print("output numbers:", sys.stdout.fileno(), sys.stderr.fileno())

### Cell 5: PRE-FLIGHT CHECK (about 15 seconds). Run it BEFORE Cell 6, never after: once the model is loaded the GPU is in use and the check will say so, which is not a fault. It checks that the jobs are the sixth run's, that every job has 100 steps, that the earlier runs' files are the ones this test names, that the files this test uses are unchanged, that the expressions are the pinned ones and the oracle's reply is inside each, that the host reader and the sentences are as written, that a first request can be built for each cell, that the **GPU power can be read**, and that the GPU is a free L4. It stops with RED if any check fails: then do not run the next cells, send me the printout

In [ ]:
from axi.experiments import provider_side_big as B
res = B.preflight()
green = B.print_preflight(res)
assert green, "PRE-FLIGHT IS RED: do not run the next cells. Send me the printout above."

### Cell 6: load the model and the engine with one expression per request (downloads about 9.4 GB once; about 8 to 15 minutes). It prints the engine's own numbers. If it fails with a red error, stop and send me the red text (if the red text says `fileno`, run Cell 4b first; if it says `StructuredOutputsParams`, send it to me and do nothing else; if it says out of memory, send it to me and do nothing else)

In [ ]:
from axi.experiments import provider_side_typed as T
from axi.experiments import provider_side_long as L
from axi.experiments import provider_side_big as B
from axi.experiments import q_measure as Q
eng = B.BigEngine()
info = eng.info(); print(json.dumps({k: v for k, v in info.items() if k != "constraint"}, indent=1))
print("constraint:", info["constraint"]["kind"], "base expression with a name of at most", info["constraint"]["name_cap"], "characters", info["constraint"]["regex_sha256"][:16], "(each request carries its own expression)")
eng.warm_up()
print("cache size the engine will use:", info["capacity_tokens"], "tokens (", info["capacity_source"], ")")

def go(label):
    """Runs one cell unless its file is already there; saves it and copies it to Drive. A cell that was cut off starts again."""
    name = f"axi_provider_b_{label}.json"; K, first, arm, mode, shape = B.CELLS[label]
    print(f"=== {label}: jobs {first} to {first + K - 1}, arm {arm}, request {mode}, grammar {shape}: {store.restore(name)}")
    if os.path.exists(store.path(name)): print("   already finished, skipping"); return
    t0 = time.time()
    def progress(i): print(f"   round {i}   {time.time() - t0:6.0f} s", end="\r")
    c = B.run_cell(eng, label, progress=progress)
    B.save_cell(c, store.path(name)); store.persist(name)
    s = B.summarize_cell(c); f = lambda x, d=1: "-" if x is None else f"{x:.{d}f}"
    print(f"\n   done: {s['steps_done']} steps, {s['wall_seconds']:.0f} s, {f(s['seconds_per_step'], 3)} s/step, exact steps {s['exact_share'] * 100:.1f} %, mean power {f(s['mean_w'])} W, {f(s['joules_per_exact_step'], 2)} J per exact step")

### Cell 7: SAFETY CHECK (about 3 minutes). 3 jobs of 3 steps of each of the seven cells. It prints what the engine answers and the GPU power it read. **A wrong answer is not a STOP** (that is what the test is for). If it says STOP, do not run the next cells: send me the printout

In [ ]:
smoke = {}
for lab in B.ORDER:
    sess = B.make_sessions(lab, 3)[:3] if B.CELLS[lab][0] >= 3 else B.make_sessions(lab, 3)
    smoke[lab] = L.run_sessions(eng, sess)
for name, sm in smoke.items():
    print(name)
    for x in sm["sessions"][0]["log"]: print(f"  step {x['t']}: prompt {x['n_prompt']:5d}  cached {x['n_cached']}  new {x['n_new']:3d}  {x['text'][:50]!r}")
allrep = [(x["parsed"], x["n_cached"], x["in_language"], x["n_new"]) for sm in smoke.values() for s in sm["sessions"] for x in s["log"]]
ok_cached = all(c is not None for _, c, _, _ in allrep)
ok_parse = all(p for p, _, _, _ in allrep)
ok_lang = all(l for _, _, l, _ in allrep)
ok_long = all(n < 96 for _, _, _, n in allrep)
m = B.PowerMeter().start(); time.sleep(2.0); pw = m.stop()
ok_power = pw["samples"] >= 2 and pw["mean_w"] is not None
print(); print("GPU power read:", pw["samples"], "readings, mean", pw["mean_w"], "W; request times reported by the engine:", [r for r in eng.request_metrics[-1:]])
print()
print("STOP: this vLLM version does not report cached tokens. Send me the printout and the version above." if not ok_cached else
      "STOP: a reply of the safety cell was refused by the strict parser, so the constraint is not working. Send me the printout." if not ok_parse else
      "STOP: a reply of the safety cell is outside the expression of its own request, so one expression per request is not working. Send me the printout." if not ok_lang else
      "STOP: a reply of the safety cell used all 96 new tokens. Send me the printout." if not ok_long else
      "STOP: the GPU power could not be read, so no energy can be measured. Send me the printout." if not ok_power else
      "OK: cached counts are reported, every reply is well formed and inside the expression of its request, the replies end and the GPU power is read. You can go on.")

### Cell 8: the seven cells, in this order: `P0`, `PN` (32 jobs), `P4`, `PC` (4 jobs), `X0`, `XW`, `XR` (32 jobs). About 60 to 120 minutes in all. A counter shows the round; a 32-job cell takes about 10 to 20 minutes. Do not close the tab. You can stop between cells: finished cells are saved

In [ ]:
for label in B.ORDER: go(label)

### Cell 9: put the finished cells together, print the table (with the power and the energy), the split by pool and by what the host read, the first misses and the 21 reading-rule lines, and download your file. You can run it at any time, with whatever cells are finished

In [ ]:
from google.colab import files
paths = sorted(glob.glob(store.path("axi_provider_b_*.json")))
print(len(paths), "cells:", [os.path.basename(p) for p in paths])
m = B.merge_files(paths, store.path("axi_provider_big.json")); store.persist("axi_provider_big.json")
S = B.summarize_all(m["cells"]); B.print_cells(S); B.print_split(S); B.print_misses(S); B.print_verdict(B.verdict_big(m["cells"]))
print(); print(Q.render_table({"Q = O / M of the cells of this run": [Q.q_row(c) for c in m["cells"]]}))
files.download(store.path("axi_provider_big.json"))

### Done
File downloaded: `axi_provider_big.json`. Upload it to the chat, with the cell files from your Drive folder `axi_provider_big` (the `axi_provider_b_*.json` files), a screenshot of the printed table, of the split and of the 21 reading-rule lines, the printout of Cell 5 and of Cell 7. Do not paste any keys anywhere.